# 03 — Processed dataset visualization

**Purpose:** Review nine labeled examples, split counts and WBC box geometry before training.

**Required inputs:** Completed notebook 02 and its processed dataset.

**Saved outputs:** `outputs/figures/processed_wbc_examples.png`, `outputs/figures/wbc_box_dimensions.png`, validation report.

## 1. Load the central configuration

The root is found from the notebook location or the installed package. Set `LEUKEMIA_YOLO_CONFIG` only for synthetic smoke tests or an alternate config.

In [ ]:
from pathlib import Path
import os, sys
search = [Path.cwd(), *Path.cwd().parents]
if os.environ.get("LEUKEMIA_YOLO_ROOT"):
    search.insert(0, Path(os.environ["LEUKEMIA_YOLO_ROOT"]))
for candidate in search:
    if (candidate / "config.example.yaml").is_file() and (candidate / "src" / "config.py").is_file():
        sys.path.insert(0, str(candidate))
        break
from src.config import load_config, ensure_outputs
cfg = load_config()
out = ensure_outputs(cfg)
os.chdir(cfg["root"])
print("Project root:", cfg["root"])
print("Configuration:", cfg["config_path"])

## 2. Validate all images and labels

Reject invalid classes, missing labels, source mismatch, overlapping splits and identical cross-split pixels.

In [ ]:
from src.validation import validate_processed
import pandas as pd
validation = validate_processed(cfg["processed_data_path"], out / "validation_report.json")
display(pd.DataFrame(validation["splits"]).T)
assert all(v["wbc_boxes"] > 0 for v in validation["splits"].values())

## 3. Plot WBC examples and box sizes

Visual review remains necessary: verify green boxes surround leukocytes and no RBC-only boxes slipped in.

In [ ]:
from src.visualization import processed_gallery, box_distribution
display(processed_gallery(cfg["processed_data_path"], out / "figures" / "processed_wbc_examples.png", count=9, seed=cfg["seed"]))
display(box_distribution(validation["box_dimensions"], out / "figures" / "wbc_box_dimensions.png"))
display(pd.DataFrame(validation["box_dimensions"]).describe())

## 4. Training readiness

Programmatic integrity passes; inspect the image grid before starting GPU work.

In [ ]:
assert validation["ready"]
assert (out / "figures" / "processed_wbc_examples.png").is_file()
print("PASS — data checks complete. Proceed to notebook 04 after visually reviewing boxes.")

## Next notebook

Open **04_YOLOv8_Training.ipynb** on the CUDA laptop.